# 01 · Read EddyPro FLUXNET CSVs → Parquet

First processing step. Each EddyPro flux output (FLUXNET CSV format, one file
per [processing version](../docs/processing-versions.md)) is read with `diive`
and written back out as a Parquet file for fast, type-safe reloading in the
downstream analysis.

- **Input:** `data/00-eddypro_fluxes_level-1/` — the raw EddyPro FLUXNET CSVs.
- **Output:** `data/01-eddypro_fluxes_level-1_parquet/` — one `.parquet` per version.

The reader is the same one used in diive's example data
(`load_exampledata_EDDYPRO_FLUXNET_CSV_30MIN`): `ReadFileType` with
`filetype='EDDYPRO-FLUXNET-CSV-30MIN'`. Saving uses `diive.core.io.files.save_parquet`.

## Imports

In [ ]:
import re
from datetime import datetime
from pathlib import Path

from diive.core.io.filereader import ReadFileType
from diive.core.io.files import save_parquet

NB_START = datetime.now()  # notebook start time (reported in the last cell)

## Configuration

In [3]:
# Input / output folders (relative to the notebooks/ directory).
INDIR = Path("../data/00-eddypro_fluxes_level-1")
OUTDIR = Path("../data/01-eddypro_fluxes_level-1_parquet")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Discover the input CSVs.
csv_files = sorted(INDIR.glob("eddypro_*_adv.csv"))
print(f"Found {len(csv_files)} flux file(s):")
for f in csv_files:
    print(f"  {f.name}")

Found 6 flux file(s):
  eddypro_LGR-1_FR-20260521-103649_fluxnet_2026-05-21T234627_adv.csv
  eddypro_LGR-2R_FR-20260527-195011_fluxnet_2026-05-28T092652_adv.csv
  eddypro_LGR-3_2021_2_2022_1_LGR_eddypro_CH-CHA_FR-20240809-181345_fluxnet_2024-08-10T185331_adv.csv
  eddypro_QCL-1_FR-20260521-103816_fluxnet_2026-05-22T021605_adv.csv
  eddypro_QCL-2R_FR-20260527-194920_fluxnet_2026-05-28T112056_adv.csv
  eddypro_QCL-3_2020_4_5_2021_1_QCL_eddypro_CH-CHA_FR-20240809-181351_fluxnet_2024-08-10T220247_adv.csv


## Version codes

In [4]:
def version_code(filename: str) -> str:
    """Extract the processing-version code (e.g. 'LGR-1', 'QCL-2R') from a filename."""
    match = re.search(r"eddypro_([A-Z]+-\d+R?)_", filename)
    if not match:
        raise ValueError(f"Could not extract a version code from: {filename}")
    return match.group(1)


# Map each input file to its version code (the output Parquet name).
for f in csv_files:
    print(f"{version_code(f.name):>8}  <-  {f.name}")

   LGR-1  <-  eddypro_LGR-1_FR-20260521-103649_fluxnet_2026-05-21T234627_adv.csv
  LGR-2R  <-  eddypro_LGR-2R_FR-20260527-195011_fluxnet_2026-05-28T092652_adv.csv
   LGR-3  <-  eddypro_LGR-3_2021_2_2022_1_LGR_eddypro_CH-CHA_FR-20240809-181345_fluxnet_2024-08-10T185331_adv.csv
   QCL-1  <-  eddypro_QCL-1_FR-20260521-103816_fluxnet_2026-05-22T021605_adv.csv
  QCL-2R  <-  eddypro_QCL-2R_FR-20260527-194920_fluxnet_2026-05-28T112056_adv.csv
   QCL-3  <-  eddypro_QCL-3_2020_4_5_2021_1_QCL_eddypro_CH-CHA_FR-20240809-181351_fluxnet_2024-08-10T220247_adv.csv


## Read each CSV and save as Parquet

`ReadFileType` returns the data (`data_df`) and a metadata table (`metadata_df`,
the variable units). The data frame is written to Parquet named after the
version code, so downstream code can load e.g. `LGR-1.parquet` directly.

In [5]:
saved = {}
for f in csv_files:
    code = version_code(f.name)
    print(f"\n=== {code} ===")

    loaddatafile = ReadFileType(
        filetype="EDDYPRO-FLUXNET-CSV-30MIN",
        filepath=str(f),
        data_nrows=None,
    )
    data_df, metadata_df = loaddatafile.get_filedata()
    print(f"  read {data_df.shape[0]} rows x {data_df.shape[1]} cols")

    filepath = save_parquet(filename=code, data=data_df, outpath=str(OUTDIR))
    saved[code] = filepath


=== LGR-1 ===


> Reading file eddypro_LGR-1_FR-20260521-103649_fluxnet_2026-05-21T234627_adv.csv ...

F:\dev\diive\diive\core\io\filereader.py:550: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[_temp_parsed_index_col] = ''


  read 7805 rows x 484 cols


> Saved file ..\data\01-eddypro_fluxes_level-1_parquet\LGR-1.parquet (0.676 seconds).

...


=== QCL-3 ===


> Reading file 
eddypro_QCL-3_2020_4_5_2021_1_QCL_eddypro_CH-CHA_FR-20240809-181351_fluxnet_2024-08-10T220247_adv.csv ...

F:\dev\diive\diive\core\io\filereader.py:550: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[_temp_parsed_index_col] = ''


  read 20790 rows x 482 cols


> Saved file ..\data\01-eddypro_fluxes_level-1_parquet\QCL-3.parquet (1.910 seconds).

## Verify

Reload one Parquet file to confirm the round-trip and inspect a few of the
study-relevant columns (fluxes + time-lag diagnostics).

In [ ]:
from diive.core.io.files import load_parquet

print("Saved Parquet files:")
for code, path in saved.items():
    print(f"  {code}: {path}")

check = load_parquet(filepath=saved["LGR-1"])
cols = [c for c in ["FN2O", "FCH4", "N2O_TLAG_USED", "CH4_TLAG_USED"] if c in check.columns]
print(f"\nLGR-1: {check.shape[0]} rows x {check.shape[1]} cols")
check[cols].describe()

## Runtime

In [ ]:
NB_END = datetime.now()
print(f"Start:    {NB_START:%Y-%m-%d %H:%M:%S}")
print(f"End:      {NB_END:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:  {NB_END - NB_START}")